In [1]:
%matplotlib inline

In [19]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
import sys
print(sys.executable)
import sys
sys.path.insert(0, "../src")
from evaluation import mean_per_user_f1, search_threshold
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from models import groupkfold_evaluate
from sklearn.pipeline import make_pipeline
from models import tune_lightgbm
from models import save_model_artifacts
import lightgbm as lgb


/opt/anaconda3/bin/python


# Instacart Market Basket Analysis

## Modelling

This notebook loads the prepared feature matrix and trains the classifier lineup along an inductive bias axis: frequency heuristic floor, logistic regression, LightGBM, MLP. The matrix carries a user-level train/test split; the test cohort is sealed until final evaluation. Hyperparameter tuning uses GroupKFold on user_id within the train split, so no user appears in both training and validation of any fold.

In [3]:
PROCESSED_DIR = Path("../data/processed")
matrix = pd.read_parquet(PROCESSED_DIR / "feature_matrix.parquet")

train = matrix[matrix["split"] == "train"].copy()
test = matrix[matrix["split"] == "test"].copy()

print(f"Train: {train.shape}, users: {train['user_id'].nunique():,}")
print(f"Test:  {test.shape}, users: {test['user_id'].nunique():,}")
print(f"Train positive rate: {train['y'].mean():.4f}")

Train: (6359442, 20), users: 98,406
Test:  (2115219, 20), users: 32,803
Train positive rate: 0.0979


In [4]:
# Feature columns: everything except keys, label, and the split marker.
non_feature_cols = ["user_id", "product_id", "y", "split"]
feature_cols = [c for c in matrix.columns if c not in non_feature_cols]

print(f"{len(feature_cols)} feature columns:")
for c in feature_cols:
    print(f"  {c}")

16 feature columns:
  count_up
  f_up
  recency_up
  mean_cart_position_up
  N_u
  mean_basket_size_u
  mean_interval_u
  var_interval_u
  interval_censored_u
  reorder_ratio_u
  unique_products_u
  r_p_shrunk
  unique_buyers_p
  mean_cart_position_p
  r_aisle
  r_dept


### Frequency Heuristic Floor

The no-learning baseline. The single feature $f_{u,p}$ (fraction of a user's prior orders containing the product) is thresholded directly: predict reorder if $f_{u,p} \geq \tau$. The threshold is tuned on the training split to maximise mean per-user F1, using the same procedure every later model uses, so the comparison is fair. This floor's F1 is the reference $\Delta$ is measured against: every learned model must beat it to justify its added complexity. The Mann-Whitney test already showed $f_{u,p}$ separates the classes with effect size 0.546, so a strong floor is expected; the open question is how much structure adds on top.

In [5]:

floor_search = search_threshold(
    user_ids=train["user_id"].values,
    y_true=train["y"].values,
    y_score=train["f_up"].values,
)

print(f"Frequency floor best threshold tau*: {floor_search['best_threshold']:.2f}")
print(f"Frequency floor mean per-user F1 (train): {floor_search['best_f1']:.4f}")

Frequency floor best threshold tau*: 0.26
Frequency floor mean per-user F1 (train): 0.3281


#### Findings: Frequency Heuristic Floor

Thresholding $f_{u,p}$ directly gives an optimal cutoff $\tau^* = 0.26$ and mean per-user F1 of 0.3281 on the training split. The rule is interpretable: predict reorder for any product appearing in at least roughly a quarter of a user's prior orders.

This is the floor every learned model is measured against, $\Delta = F_1^{\text{model}} - 0.3281$. Two readings frame its strength. Against the recall ceiling of 0.5986 rather than a naive 1.0, the floor already captures about 55% of the achievable F1 using a single feature and no learning, confirming the central thesis that reorder behaviour is dominated by raw recurrence. And the threshold sits well below 0.5, as the F1 geometry predicts when positives are rare ($\rho \approx 0.10$): $\tau^* \approx F_1^*/2$ would put it near 0.16, and the observed 0.26 is in that low-cutoff regime, a consequence of the metric rather than a tuning accident.

### Logistic Regression: The Linear Floor

The first learned model and the linear rung of the inductive bias axis. Where the frequency heuristic thresholded a single feature, logistic regression combines all 16 features through a linear decision boundary:

$$P(y_{u,p} = 1) = \sigma\left(\mathbf{w}^\top \mathbf{x}_{u,p} + b\right) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x}_{u,p} + b)}}$$

Its job is to measure how much signal lives in a hyperplane: how much does linearly combining all features beat thresholding the single best one. Features are standardised before fitting, since logistic regression is sensitive to feature scale, and the scaling is fit within each fold only, never on held-out data, so no information leaks across the fold boundary. The regularisation strength $C$ is tuned by GroupKFold cross-validation, the same user-grouped folds every model uses.

In [6]:
X_train = train[feature_cols]
y_train = train["y"].values
groups_train = train["user_id"].values

# Tune C via GroupKFold. Each pipeline scales within-fold then fits logistic regression.
C_values = [0.01, 0.1, 1.0, 10.0]
lr_results = {}

for C in C_values:
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000))
    res = groupkfold_evaluate(pipe, X_train, y_train, groups_train, n_splits=5)
    lr_results[C] = res
    print(f"C={C:>5}: mean F1 = {res['mean_f1']:.4f} +/- {res['std_f1']:.4f}")

C= 0.01: mean F1 = 0.3593 +/- 0.0017
C=  0.1: mean F1 = 0.3593 +/- 0.0017
C=  1.0: mean F1 = 0.3593 +/- 0.0017
C= 10.0: mean F1 = 0.3593 +/- 0.0017


In [7]:


# Fit on a sample with two extreme C values, compare the actual probabilities.
sample = train.sample(50000, random_state=0)
Xs = sample[feature_cols]
ys = sample["y"].values

probs = {}
for C in [0.01, 10.0]:
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000))
    pipe.fit(Xs, ys)
    probs[C] = pipe.predict_proba(Xs)[:, 1]

import numpy as np
print(f"Mean abs difference in probabilities between C=0.01 and C=10: {np.abs(probs[0.01] - probs[10.0]).mean():.6f}")
print(f"Max abs difference: {np.abs(probs[0.01] - probs[10.0]).max():.6f}")

Mean abs difference in probabilities between C=0.01 and C=10: 0.005812
Max abs difference: 0.181774


### Final Logistic Regression: Fit and Verify

C=1.0 is used for the final model, since the sweep showed C has no effect on thresholded F1. Before saving, convergence is verified directly rather than assumed: max_iter is a solver ceiling, not a tunable hyperparameter, so the check is whether the solver actually finished, not a search over its value.

In [12]:
import warnings

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    final_logreg = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
    final_logreg.fit(train[feature_cols], train["y"].values)

    convergence_warnings = [x for x in w if "converge" in str(x.message).lower()]
    if convergence_warnings:
        print("WARNING: did not converge within max_iter=1000")
    else:
        print("Converged cleanly within max_iter=1000, no warning raised.")

print(f"n_iter_ actually used: {final_logreg.named_steps['logisticregression'].n_iter_}")

Converged cleanly within max_iter=1000, no warning raised.
n_iter_ actually used: [17]


In [13]:
final_logreg = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
final_logreg.fit(train[feature_cols], train["y"].values)

print("Final logistic regression fitted on full training set.")

Final logistic regression fitted on full training set.


In [14]:
# Use the C=1.0 result's mean threshold across folds as the model's tuned tau*.
logreg_cv = lr_results[1.0]  # the C=1.0 entry from your earlier sweep
tau_logreg = float(np.mean(logreg_cv["fold_threshold"]))

save_model_artifacts(
    model_dir="../models/logistic_v1",
    model=final_logreg,
    model_kind="sklearn",
    feature_cols=feature_cols,
    threshold=tau_logreg,
    cv_results=logreg_cv,
    extra={"C": 1.0},
)

Saved model artifacts to ../models/logistic_v1
  model_kind: sklearn
  features: 16
  threshold: 0.1620
  mean_f1: 0.3593


PosixPath('../models/logistic_v1')

#### Findings: Logistic Regression

Logistic regression achieves mean per-user F1 of 0.3593 under 5-fold GroupKFold, against the frequency floor of 0.3281, a margin of $\Delta = 0.031$. Linearly combining all 16 features beats thresholding the single best feature, but modestly: most of the predictable structure was already captured by $f_{u,p}$ alone, consistent with the central thesis that reorder behaviour is dominated by raw recurrence.

The score is identical across $C \in \{0.01, 0.1, 1.0, 10.0\}$. This is not regularisation having no effect, the underlying probabilities do shift with $C$ (mean absolute change 0.006, up to 0.18 on individual predictions). Rather, because the decision threshold is re-tuned per fold, those probability shifts are absorbed by the threshold adapting, leaving the final thresholded per-user F1 unchanged. The thresholded metric is therefore robust to regularisation strength, and $C$ is not a meaningful lever for this model under this evaluation.

### LightGBM: The Primary Model

The tabular workhorse and expected best performer on this axis, able to capture axis-aligned nonlinear interactions, recency combined with frequency, product base rate combined with user propensity, without them being hand-engineered as explicit interaction terms. This is the model that directly tests whether logistic regression was underfitting: if LightGBM jumps meaningfully above 0.359, nonlinear structure was sitting there unused; if it stays close, the residual is genuinely close to irreducible given this feature set.

Hyperparameters are searched via RandomizedSearchCV scored on AUC (a standard, library-native metric that gives a stable read on overall ranking quality), then the winning configuration is evaluated on the true task metric, mean per-user F1 with per-fold threshold tuning, via the same GroupKFold procedure used for logistic regression.

In [18]:
lgbm_results = tune_lightgbm(
    X=train[feature_cols],
    y=train["y"].values,
    groups=train["user_id"].values,
    n_iter=8,
    n_splits_search=3,
)

print(f"Best params: {lgbm_results['best_params']}")
print(f"Search AUC: {lgbm_results['search_auc']:.4f}")
print(f"Per-fold F1: {[round(x,4) for x in lgbm_results['f1_eval']['fold_f1']]}")
print(f"Mean F1: {lgbm_results['f1_eval']['mean_f1']:.4f} +/- {lgbm_results['f1_eval']['std_f1']:.4f}")

Best params: {'reg_lambda': 0.0, 'reg_alpha': 0.0, 'num_leaves': 127, 'min_child_samples': 50, 'learning_rate': 0.03}
Search AUC: 0.8305
Per-fold F1: [0.3751, 0.3724, 0.3709, 0.3754, 0.3731]
Mean F1: 0.3734 +/- 0.0017


The first search found `num_leaves=127`, the upper edge of the initial grid, which is a signal the optimal value may lie beyond what was searched. The grid is widened to include 255 and 511 leaves, and the search repeated with more iterations to verify whether additional capacity genuinely helps or whether 127 was already near the true optimum.

In [21]:
wider_grid = {
    "num_leaves": [63, 127, 255, 511],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "min_child_samples": [20, 50, 100, 200],
    "reg_lambda": [0.0, 0.1, 1.0, 5.0],
    "reg_alpha": [0.0, 0.1, 1.0],
}

lgbm_results_wide = tune_lightgbm(
    X=train[feature_cols],
    y=train["y"].values,
    groups=train["user_id"].values,
    param_distributions=wider_grid,
    n_iter=15,
    n_splits_search=3,
)

print(f"Best params: {lgbm_results_wide['best_params']}")
print(f"Search AUC: {lgbm_results_wide['search_auc']:.4f}")
print(f"Per-fold F1: {[round(x,4) for x in lgbm_results_wide['f1_eval']['fold_f1']]}")
print(f"Mean F1: {lgbm_results_wide['f1_eval']['mean_f1']:.4f} +/- {lgbm_results_wide['f1_eval']['std_f1']:.4f}")

Best params: {'reg_lambda': 5.0, 'reg_alpha': 0.1, 'num_leaves': 63, 'min_child_samples': 20, 'learning_rate': 0.03}
Search AUC: 0.8307
Per-fold F1: [0.3754, 0.3721, 0.371, 0.3758, 0.3728]
Mean F1: 0.3734 +/- 0.0019


In [17]:
print(train[feature_cols].shape)
print(train[feature_cols].dtypes)

(6359442, 16)
count_up                   int16
f_up                     float32
recency_up                 int16
mean_cart_position_up    float32
N_u                        int32
mean_basket_size_u       float32
mean_interval_u          float32
var_interval_u           float32
interval_censored_u         int8
reorder_ratio_u          float32
unique_products_u          int32
r_p_shrunk               float32
unique_buyers_p            int32
mean_cart_position_p     float32
r_aisle                  float32
r_dept                   float32
dtype: object


#### Findings: LightGBM Hyperparameter Search

Two searches were run. The first used a narrower grid and found `num_leaves=127`, the upper boundary of that grid, which is a diagnostic signal that the true optimum might lie outside what was searched. A second search widened the grid to include 255 and 511 leaves and added more iterations.

The wider search resolved the boundary question cleanly: `num_leaves=63` won, a value *below* the original boundary, with stronger regularisation (`reg_lambda=5.0`, `reg_alpha=0.1`) than the first search found. Both searches converged to the same mean per-user F1 of 0.3734. The grid for `num_leaves` followed a geometric progression (63, 127, 255, 511, corresponding roughly to balanced binary trees of depth 6 through 9), which is the natural spacing for this parameter since model complexity scales multiplicatively with leaf count, not additively.

The convergence of F1 across a wide range of tree capacities, with the search preferring a simpler, more regularised model when given the choice, is itself a finding: the bottleneck is feature informativeness, not model capacity. More leaves does not help because the signal available in the 16 engineered features is already mostly captured at modest complexity. This is consistent with the diminishing-returns pattern seen across the full inductive bias axis: floor (0.3281) → logistic regression (0.3593) → LightGBM (0.3734), each step adding less than the one before, the residual approaching the irreducible component of reorder behaviour given this feature set.

In [23]:
best_params = lgbm_results_wide["best_params"]

final_lgbm = lgb.LGBMClassifier(
    n_estimators=1000,
    random_state=42,
    verbosity=-1,
    **best_params,
)
final_lgbm.fit(train[feature_cols], train["y"].values)
print(f"Final LightGBM fitted with: {best_params}")

Final LightGBM fitted with: {'reg_lambda': 5.0, 'reg_alpha': 0.1, 'num_leaves': 63, 'min_child_samples': 20, 'learning_rate': 0.03}


In [24]:
tau_lgbm = float(np.mean(lgbm_results_wide["f1_eval"]["fold_threshold"]))

save_model_artifacts(
    model_dir="../models/lgbm_v1",
    model=final_lgbm,
    model_kind="lightgbm",
    feature_cols=feature_cols,
    threshold=tau_lgbm,
    cv_results=lgbm_results_wide["f1_eval"],
    extra={
        "best_params": best_params,
        "search_auc": lgbm_results_wide["search_auc"],
        "note": "wider grid confirmed num_leaves=63 with stronger regularization, same F1 as narrower search"
    },
)

Saved model artifacts to ../models/lgbm_v1
  model_kind: lightgbm
  features: 16
  threshold: 0.1920
  mean_f1: 0.3734


PosixPath('../models/lgbm_v1')